# ZuCo paper-style analysis: original agent-labelled word fixations
Select **T4 GPU → Run all**, using the same Zyion Drive account. No new native EEG extraction or cleaning.

**Labels:** original `word_labels.csv` agent review. Keep positive/negative words only when lexical and word-in-context labels agree, with no alignment warning. No whole-sentence agreement or content-word filter. Agent labels are not human-consensus gold labels. Neutral is excluded for this binary paper comparison.

**Important coverage limit:** only fixations already present in the completed **SST-selected** paper archive can be used. This is the intersection with our earlier annotations, not a complete agent-labelled cohort. It can miss agent-positive/negative words whose SST label was neutral, unavailable, or otherwise excluded. The inventory reports this explicitly. The original agent SR pool has 495 positive and 442 negative source word occurrences before reader availability/rejection.

All archived fixations of selected words are candidates, including rereading. The original first-fixation-only restriction is not used here. New subset labels replace the SST target label, and positive/negative classes are rematched. Existing full-reader cleaning is retained. Original negation exclusions are inherited; contractions are not newly handled.

**Storage:** separate `zuco_agent_paper_subset/` result folder, hard **200 MB total cap** for statistics, compact checkpoints, plots and decoding results. EEG/features stay only on runtime disk. Completed work resumes; no old results or trials are deleted. This includes 5000 TCT/1000 cluster permutations, mixed models and linear decoding; it may take substantial time but avoids native reconstruction.


## Connect Drive and pinned code

In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
RUN=THESIS/'Results/word_sentiment_eeg/zuco_paper_figures_recovered'
OUT=THESIS/'Results/word_sentiment_eeg/zuco_agent_paper_subset'
REPO=Path('/content/word-sentiment-eeg-agent-subset')
CODE_REVISION='3e165ec7e4cba58f30f7d57128be4e180d9faa8d'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-paper.txt')],check=True)
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# A mounted Drive filesystem disk_usage value can reflect the runtime volume;
# use the authenticated Drive quota instead.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();quota=response.json()['storageQuota']
print('Quota account:',response.json().get('user',{}).get('emailAddress','unavailable'))
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a finite storage quota; verify free space before starting.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
print(f'Account free space: {FREE_DRIVE_BYTES/1e9:.2f} GB. Workflow checks the additional space needed before source reads.')
print('Destination:',OUT)

used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES < max(0,200_000_000-used):
    raise RuntimeError('Need up to 200 MB free for all new follow-up outputs; existing results unchanged.')


## Run selection, paper-style statistics and decoding

In [ ]:
import torch
if not torch.cuda.is_available():raise RuntimeError('Select T4 GPU before running this notebook.')
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/zuco_agent_paper_subset.py'),'--thesis-root',str(THESIS)]
run_with_progress(command,Path('/content/agent-paper-subset-status.json'),activity='Agent-labelled paper subset: saved EEG only')


## 1. Source words versus available and retained EEG trials

In [ ]:
inv=json.loads((OUT/'selection/inventory.json').read_text())
print(inv['warning'])
print('Agent-labelled source word occurrences, NOT multiplied by readers:',inv['source_word_occurrences'])
readers=pd.DataFrame(inv['readers'])
display(readers)
display(readers[['parent_accepted','available_agent_eligible','subset_negative','subset_positive']].sum().rename('summed_across_readers').to_frame())
print('Availability by source occurrence and reader:')
display(pd.DataFrame(inv['coverage']))
print('Full selected/excluded event ledger with original and agent labels:',OUT/'selection/trial_selection.csv.gz')


## 2. Selection and available controls

In [ ]:
display(Image(filename=str(OUT/'figures/subset_selection_controls.png')))
balance=pd.DataFrame(json.loads((OUT/'analysis/covariate_balance.json').read_text()))
display(balance[balance.reader.eq('ALL_TRIALS')])
print('Subset-specific eye-velocity waveforms cannot be reconstructed from the saved accepted EEG archive. The original full-cohort eye controls are not reused.')


## 3. Raw fixation-related potentials — paper A–F arrangement

In [ ]:
display(Image(filename=str(OUT/'figures/figure_2_raw_frp.png')))


## 4. Cleaned fixation-related potentials — paper A–F arrangement

In [ ]:
display(Image(filename=str(OUT/'figures/figure_3_cleaned_frp.png')))


## 5. Corrected clusters and additional reader-level window tests

In [ ]:
summary=json.loads((OUT/'analysis/summary.json').read_text())
display(pd.DataFrame([{k:v for k,v in summary.items() if k!='scope'}]))
print(summary['scope'])
if any(summary['nonconverged_features']):print('WARNING: some model features did not converge; inspect before interpreting p-values.')
clusters=pd.DataFrame(json.loads((OUT/'analysis/clusters.json').read_text()))
if len(clusters):display(clusters.sort_values('p').head(12))
else:print('No observed supra-threshold clusters.')
display(pd.DataFrame(json.loads((OUT/'analysis/window_tests.json').read_text())))
print('Window tests are additional exploratory reader sign-flip tests, Holm-corrected over four comparisons; they are not the cluster test.')


## 6. Binary decoding: individual fixations and 20/40-trial averages

In [ ]:
results=json.loads((OUT/'classification/summary.json').read_text())
rows=[]
for s in results:
    p=s.get('pooled') or {}
    rows.append({k:v for k,v in s.items() if k!='pooled'}|{k:p.get(k) for k in ['n','accuracy','balanced_accuracy','macro_f1','auc']})
scores=pd.DataFrame(rows)
display(scores[scores.window.eq('published_224_304')].sort_values(['split','representation','average_trials','model']))
print('Chance balanced accuracy=50%. Test averages use known classes, formed disjointly after group splitting; not single-event decoding.')


## 7. Longer-window decoding and controls

In [ ]:
display(scores[scores.window.isin(['post_0_800','pre_minus200_0','metadata_only'])].sort_values(['window','split','representation','average_trials','model']))
print('Same fixed logistic/dummy experiments as the recovered-cohort follow-ups: unseen sentences, normalized word spellings, or readers. No CNN here.')
print('Outputs:',OUT)
print('Saved output size (MB):',round(sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file())/1e6,2))


## Interpretation and limitations
The raw/clean waveforms and all upstream SST-based selection, exclusions and matching come from the parent archive. New rematching happens after those earlier quality exclusions, so it cannot recreate the authors’ full agent-label selection from continuous recordings. A missing waveform is not reconstructed or replaced by a class average.

Cleaning was fitted on each full reader recording. These decoding splits therefore have transductive preprocessing and are exploratory. The same 40-ms features, logistic model, grouped splits, controls and class-known 20/40-trial averaging from the previous follow-ups are reused. They are an independent Python comparison, not exact author-code replication. Multiple analyses are reported without choosing settings to maximize significance.

To analyze **every** originally agent-selected word fixation, including ones absent from this archive, a separately authorized reconstruction from native sources would be needed. This notebook deliberately makes no such claim. It preserves the completed SST reproduction and earlier results.
